In [1]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections 

In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

`sede` no trae el nombre del departamento directamente: hay que hacer join `sede -> ciudad -> departamento`.

In [3]:
dim_sede = pd.read_sql_query('''
    SELECT
        s.sede_id,
        s.nombre          AS nombre_sede,
        s.direccion,
        s.telefono,
        s.nombre_contacto,
        c.nombre          AS ciudad,
        d.nombre          AS departamento,
        s.cliente_id
    FROM sede s
    LEFT JOIN ciudad c ON s.ciudad_id = c.ciudad_id
    LEFT JOIN departamento d ON c.departamento_id = d.departamento_id
''', co_sa)

dim_sede.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sede_id          52 non-null     int64
 1   nombre_sede      52 non-null     str  
 2   direccion        52 non-null     str  
 3   telefono         52 non-null     str  
 4   nombre_contacto  52 non-null     str  
 5   ciudad           52 non-null     str  
 6   departamento     52 non-null     str  
 7   cliente_id       52 non-null     int64
dtypes: int64(2), str(6)
memory usage: 3.4 KB


In [4]:
dim_sede.head()

,sede_id,nombre_sede,direccion,telefono,nombre_contacto,ciudad,departamento,cliente_id
0,10,FARALLONES /123,Los angeles distrito Latino,310-70000,JUAN PEREZ,CALI,VALLE DEL CAUCA,4
1,11,REMEDIOZ/ 123,Los angeles distrito Latino,310-70000,JUAN PEREZ,CALI,VALLE DEL CAUCA,4
2,13,DIME / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,CALI,VALLE DEL CAUCA,4
3,14,DESPACHOS / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,CALI,VALLE DEL CAUCA,4
4,23,POPAYAN BODEGA 28 / A,Los angeles distrito Latino,310-70000,JUAN PEREZ,POPAYAN,CAUCA,11


# Transformations

In [5]:
# mismo patron de limpieza de nulos/vacios usado en dim_estado, dim_mensajero y dim_tipo_servicio
dim_sede.replace({np.nan: 'no aplica', ' ': 'no aplica', '': 'no_aplica'}, inplace=True)
dim_sede['saved'] = date.today()

In [6]:
dim_sede.describe(include='all')

,sede_id,nombre_sede,direccion,telefono,nombre_contacto,ciudad,departamento,cliente_id,saved
count,52.000000,52,52,52,52,52,52,52.000000,52
unique,NaN,49,1,1,1,6,3,NaN,1
top,NaN,NUEVA HEMATO,Los angeles distrito Latino,310-70000,JUAN PEREZ,CALI,VALLE DEL CAUCA,NaN,2026-06-24
freq,NaN,3,52,52,52,45,49,NaN,52
mean,28.096154,NaN,NaN,NaN,NaN,NaN,NaN,10.307692,NaN
std,17.730288,NaN,NaN,NaN,NaN,NaN,NaN,6.276437,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,NaN
25%,13.750000,NaN,NaN,NaN,NaN,NaN,NaN,5.000000,NaN
50%,27.000000,NaN,NaN,NaN,NaN,NaN,NaN,10.000000,NaN
75%,41.250000,NaN,NaN,NaN,NaN,NaN,NaN,11.000000,NaN


# load

In [7]:
# NOTA: cambiado de if_exists='replace' a 'append' porque la tabla ahora se crea por DDL
# (ver sqlscripts.yml) con su PK real. 'replace' borraria esa tabla y la reemplazaria sin
# restricciones. Antes de correr este notebook, asegurate que main.py ya ejecuto el DDL
# (o ejecuta el script de sqlscripts.yml manualmente si la tabla aun no existe).
dim_sede.to_sql('dim_sede', etl_conn, if_exists='append', index_label='key_dim_sede')

52